[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardoberta/regressione/blob/main/03_lo_scout_migliora.ipynb)

# Lo scout migliora

Uno scout umano non giudica un calciatore guardando un solo numero. In questo episodio passiamo da una regressione con una variabile a una regressione con più variabili.

> **Missione** — Confrontare un modello semplice con un modello piu' informato, usando age, overall, potential e wage_eur.


## Più di un indizio per volta

Uno scout vero non guarda mai un solo numero. Quando deve giudicare un calciatore mette insieme età, livello tecnico, prospettive di crescita, salario, e tante altre informazioni. Il nostro scout automatico, finora, ne usa una sola: l'overall. È un buon punto di partenza, ma è ovviamente limitato.

In questo episodio lo facciamo crescere. Ricarichiamo lo stesso file di prima e gli daremo da masticare *quattro* indizi contemporaneamente, per vedere se l'errore diminuisce.


In [ ]:
# Librerie necessarie per scaricare ed estrarre il dataset
import urllib.request, zipfile
from pathlib import Path

# Librerie per l'analisi dei dati e la visualizzazione
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Link al dataset e percorsi locali
DATA_URL = (
    "https://www.dropbox.com/scl/fi/0l5n46qjwcd5moj3w7d8p/"
    "fifa.zip?rlkey=rcqhagvq5ttlvna5t5r3vn1bm&st=uzplzs5o&dl=1"
)
ZIP_PATH = Path("fifa.zip")
DATA_DIR = Path("fifa_data")

if not ZIP_PATH.exists():
    print("Scarico il dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not list(DATA_DIR.rglob("*.csv")):
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

csv_files = (
    list(DATA_DIR.rglob("players_22.csv"))
    or list(DATA_DIR.rglob("*players*.csv"))
)
raw_data = pd.read_csv(csv_files[0], low_memory=False)
print(f"Caricato: {raw_data.shape[0]} giocatori, {raw_data.shape[1]} colonne")
raw_data.head()


Stessa pulizia degli episodi precedenti: colonne utili, niente valori mancanti, taglio dell'1% di giocatori più costosi. Ormai è un rito che apre ogni notebook della serie.


In [ ]:
# Definiamo un sottoinsieme di colonne che ci interessano
wanted_columns = [
    "short_name", "age", "overall", "potential", "wage_eur", "value_eur",
    "club_name", "league_name", "player_positions"
]

# Selezioniamo le colonne disponibili e creiamo una copia di lavoro
dataset = raw_data[wanted_columns].copy()

# Manteniamo solo i giocatori con valore noto e positivo
dataset = dataset.dropna(subset=["value_eur", "age", "overall", "potential"])
dataset = dataset[dataset["value_eur"] > 0]

# Rimuoviamo l'1% dei giocatori più costosi (i fuoriclasse del dataset)
dataset = dataset[dataset["value_eur"] <= dataset["value_eur"].quantile(0.99)]

# Per leggibilità: il valore in milioni di euro e il salario in migliaia
dataset["value_mln_eur"] = dataset["value_eur"] / 1_000_000
if "wage_eur" in dataset.columns:
    dataset["wage_k_eur"] = dataset["wage_eur"] / 1_000

print("Forma del dataset pulito:", dataset.shape)
dataset.head(10)


Importiamo gli stessi strumenti del notebook precedente: la regressione lineare di sklearn e le tre metriche di errore (MAE, RMSE, $R^2$). Non c'è bisogno di nuovi algoritmi: il bello è che cambiando *cosa* diamo in pasto al modello, lo stesso codice diventa molto più potente.


In [ ]:
# Stessi strumenti del notebook precedente:
# - LinearRegression: il modello
# - le tre metriche per misurare l'errore
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
)


> **Regressione multipla** — Nella regressione multipla il modello usa piu' input contemporaneamente. Geometricamente non stiamo piu' adattando una retta nel piano, ma un piano o un iperpiano in uno spazio con piu' dimensioni. L'idea resta la stessa: trovare la combinazione di variabili che riduce l'errore.


### Da una retta a un iperpiano

Quando passiamo da un solo input a $p$ input $x_1, x_2, \dots, x_p$, la formula del modello cresce di pari passo:

$\displaystyle \hat{y} \;=\; w_1 x_1 + w_2 x_2 + \dots + w_p x_p + b$

Ogni variabile ha il suo peso $w_j$, che dice quanto quel singolo indizio sposta la previsione. In forma compatta, raccogliendo gli input nel vettore $\mathbf{x} = (x_1, \dots, x_p)$ e i pesi in $\mathbf{w} = (w_1, \dots, w_p)$:

$\displaystyle \hat{y} \;=\; \mathbf{w}^{\top}\mathbf{x} + b$

Geometricamente, con $1$ variabile cercavamo una **retta**; con $2$ cerchiamo un **piano**; con $p$ un **iperpiano** in uno spazio a $p$ dimensioni — qualcosa che non possiamo più disegnare ma che funziona esattamente come prima. La logica e la funzione di costo restano le stesse del notebook precedente:

$\displaystyle \mathrm{MSE}(\mathbf{w}, b) \;=\; \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$

Cambia solo il numero di parametri da imparare: $p+1$ pesi invece di $2$.


## Modello A: solo overall

Per prima cosa ricostruiamo lo scout dell'episodio precedente: quello che usa **solo l'overall**. Sarà il nostro punto di partenza, il modello da battere. Senza un riferimento, dire "il nuovo modello è meglio" non significa nulla.


In [ ]:
# Il modello A usa una sola colonna come input: l'overall
X_one = dataset[["overall"]]
y = dataset["value_mln_eur"]

# Stessa procedura del notebook precedente: creiamo, addestriamo, prevediamo
model_one = LinearRegression()
model_one.fit(X_one, y)
pred_one = model_one.predict(X_one)

# Misuriamo l'errore con le tre metriche standard
mae_one = mean_absolute_error(y, pred_one)
rmse_one = mean_squared_error(y, pred_one) ** 0.5
r2_one = r2_score(y, pred_one)

print(f"MAE modello A: {mae_one:.2f} milioni")
print(f"RMSE modello A: {rmse_one:.2f} milioni")
print(f"R^2 modello A: {r2_one:.3f}")


## Modello B: più indizi

Adesso diamo allo scout **quattro indizi** invece di uno: età, overall, potential e salario. Notate quanto è poco il codice da cambiare — letteralmente la lista delle colonne di input. Questa è una delle ragioni per cui sklearn è così diffuso: una volta capito un modello, passare da una variante all'altra costa pochissimo.


In [ ]:
# Il modello B usa quattro indizi contemporaneamente.
# Il list comprehension qui serve a essere robusti: prendiamo le
# colonne solo se esistono nel dataset.
features = [
    c for c in ["age", "overall", "potential", "wage_eur"]
    if c in dataset.columns
]
X_many = dataset[features]

# Stessa identica procedura: l'algoritmo non cambia, cambia X
model_many = LinearRegression()
model_many.fit(X_many, y)
pred_many = model_many.predict(X_many)

# Misuriamo l'errore esattamente come prima, per poter confrontare
mae_many = mean_absolute_error(y, pred_many)
rmse_many = mean_squared_error(y, pred_many) ** 0.5
r2_many = r2_score(y, pred_many)

print("Feature usate:", features)
print(f"MAE modello B: {mae_many:.2f} milioni")
print(f"RMSE modello B: {rmse_many:.2f} milioni")
print(f"R^2 modello B: {r2_many:.3f}")


> **Confronto tra modelli** — Un modello piu' ricco puo' usare piu' informazione. Se aggiungiamo variabili sensate, l'errore tende a diminuire. Pero' aggiungere variabili non e' sempre una vittoria: alcune colonne possono essere rumorose, ridondanti o persino fuorvianti.


## Confrontiamo gli errori

Mettiamo i due modelli faccia a faccia in una tabella: stessi giocatori, stesso target, due *ricette* diverse. Chi vince? E soprattutto: di quanto?


In [ ]:
# Costruiamo un piccolo DataFrame di confronto:
# una riga per modello, le tre metriche affiancate.
comparison = pd.DataFrame({
    "modello": ["A: solo overall", "B: piu' indizi"],
    "MAE_mln": [mae_one, mae_many],
    "RMSE_mln": [rmse_one, rmse_many],
    "R2": [r2_one, r2_many]
})
comparison.round(3)


## Quanto pesa ogni variabile?

Una domanda naturale, dopo aver visto che il modello B funziona meglio: nella nuova formula, **quale indizio conta di più**? Stampiamo i coefficienti del modello B, uno per ogni variabile. Sono i $w_j$ della formula della teoria di prima.


In [ ]:
# Costruiamo una tabellina con un peso per ogni feature.
# Attenzione: questi sono i coefficienti GREZZI, non ancora confrontabili
# (le variabili hanno scale molto diverse, vedi la prossima sezione).
coef = pd.DataFrame({
    "feature": features,
    "coefficiente": model_many.coef_,
})
coef.sort_values("coefficiente", ascending=False)


> **Attenzione ai coefficienti** — I coefficienti non sono sempre confrontabili direttamente, perche' le variabili hanno scale diverse. Un punto di overall e un euro di salario non hanno la stessa unita'. Per interpretazioni serie bisognerebbe normalizzare le variabili. Qui usiamo i coefficienti solo per intuire che il modello combina piu' indizi.


## Confronto onesto dei pesi: standardizzare le variabili


### Standardizzare per confrontare i pesi

Le nostre variabili hanno **scale molto diverse**: l'overall va da circa $50$ a $95$, il salario va da poche migliaia a centinaia di migliaia di euro. Quando il modello impara un peso $w_j$ per il salario, quel numero risulta minuscolo rispetto al peso dell'overall — *non perché il salario conti poco*, ma perché il salario è espresso in unità grandi. Confrontare direttamente i coefficienti è quindi fuorviante.

Il trucco standard è trasformare ogni variabile per metterle tutte sulla stessa scala:

$\displaystyle z_j \;=\; \frac{x_j - \bar{x}_j}{\sigma_j}$

dove $\bar{x}_j$ è la media e $\sigma_j$ la deviazione standard della $j$-esima variabile. Dopo la trasformazione tutte le variabili hanno **media $0$ e deviazione standard $1$**: ogni unità rappresenta "una deviazione standard". Adesso il coefficiente più grande in valore assoluto indica davvero la variabile più influente.


Applichiamo la trasformazione: ogni variabile avrà media $0$ e deviazione standard $1$. Riaddestriamo il modello sui dati standardizzati e leggiamo i nuovi pesi, finalmente confrontabili tra loro.


In [ ]:
# StandardScaler è lo strumento di sklearn che standardizza le variabili
from sklearn.preprocessing import StandardScaler

# Standardizziamo tutte le feature: media 0, deviazione standard 1 per ognuna.
# fit_transform() calcola media e std sui dati e li usa per trasformarli.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_many)

# Riaddestriamo lo stesso modello, ma sui dati standardizzati.
# Le previsioni saranno identiche al modello B, ma i pesi cambieranno scala.
model_scaled = LinearRegression()
model_scaled.fit(X_scaled, y)

# Ora i pesi sono direttamente confrontabili tra loro
pesi = (
    pd.DataFrame({
        "feature": features,
        "peso_standardizzato": model_scaled.coef_,
    })
    .sort_values(
        "peso_standardizzato",
        key=lambda s: s.abs(),
        ascending=False,
    )
)
pesi.round(3)


Una tabella di numeri è informativa, ma un grafico parla più chiaro. Visualizziamo i pesi standardizzati con un grafico a barre orizzontali. Più una barra è lunga in valore assoluto, più quella variabile è influente sul valore di mercato. Le barre verdi sono pesi positivi (al crescere della variabile cresce il valore), quelle rosse sono pesi negativi (al crescere della variabile, il valore diminuisce).


In [ ]:
# Prepariamo la figura
plt.figure(figsize=(7,4))

# Verde se il peso è positivo, rosso se è negativo:
# il colore comunica subito la direzione dell'effetto
colori = [
    "#2a9d8f" if w > 0 else "#e76f51"
    for w in pesi["peso_standardizzato"]
]

# barh = barre orizzontali (le feature stanno sull'asse y)
plt.barh(pesi["feature"], pesi["peso_standardizzato"], color=colori)

# Linea verticale a 0: separa i pesi positivi dai negativi
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Peso (su variabili standardizzate)")
plt.title("Quale variabile pesa di piu' sul valore di mercato?")

# Inverte l'asse y: la feature più importante in cima
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis="x")
plt.show()


## Lo scout automatico su giocatori casuali

Test finale dell'episodio: prendiamo 12 giocatori a caso e per ognuno mostriamo, fianco a fianco, cosa prevede il modello *solo overall* (il nostro punto di partenza) e cosa prevede il modello *più indizi*. A occhio, quanto migliora il secondo? E in quali casi la differenza è più visibile?


In [ ]:
# Estraiamo 12 giocatori a caso (random_state fissa il seed)
sample = dataset.sample(12, random_state=13).copy()

# Previsione del modello A (solo overall)
sample["pred_solo_overall"] = model_one.predict(sample[["overall"]])

# Previsione del modello B (quattro indizi)
sample["pred_piu_indizi"] = model_many.predict(sample[features])

# Mostriamo le colonne rilevanti, arrotondate a 2 decimali
sample[[
    "short_name", "age", "overall", "potential", "wage_eur",
    "value_mln_eur", "pred_solo_overall", "pred_piu_indizi"
]].round(2)


---

> **Domanda** — Trovate un giocatore per cui il modello con piu' indizi migliora molto rispetto al modello con solo overall. Quale informazione extra probabilmente lo ha aiutato?


---

> **Cosa dovremmo aver capito** — La regressione non e' solo una retta: e' un modo per costruire una funzione che trasforma informazioni in una previsione numerica. Nel prossimo episodio faremo il controllo piu' importante: il modello funziona anche su giocatori che non ha usato per imparare?
